# Taller 1 — Prueba en Google Colab

Prueba el código en un entorno distinto al computador personal (sección 3.1 del enunciado).

1. Clona el repositorio e instala las dependencias de `requirements.txt`.
2. **Sin conexión:** reconstruye la base desde `raw/` y la compara con `comparador_precios.sqlite`.
3. **En línea:** corre el scraper con un solo comercio (Locatel, unos 5 minutos) en una base aparte.

Para correrlo: *Entorno de ejecución → Ejecutar todo*.

In [ ]:
!git clone --depth 1 https://github.com/marlonjimenezv/marlon_jimenez_valencia.git
%cd marlon_jimenez_valencia/marlon_jimenez_valencia/taller_01
!python --version
!pip install -q -r requirements.txt

## 1. Modo sin conexión: reconstruir la base desde `raw/`

In [ ]:
!python scraper.py --sin-conexion --db /content/verificacion.sqlite | tail -n 6

In [ ]:
import sqlite3
import pandas as pd
import limpieza

entregada = sqlite3.connect("comparador_precios.sqlite")
nueva = sqlite3.connect("/content/verificacion.sqlite")
limpieza.cargar_correspondencia(nueva, "correspondencia.csv")
nueva.commit()

consultas = {
    "observaciones por ejecución y comercio":
        "SELECT id_ejecucion, id_comercio, COUNT(*) AS n, SUM(precio) AS suma FROM observaciones GROUP BY 1, 2 ORDER BY 1, 2",
    "anuncios":
        "SELECT id_comercio, sku_comercio, ean, principio_activo_std, unidades, contenido, comparable FROM anuncios ORDER BY 1, 2",
    "precios comparables":
        "SELECT o.id_ejecucion, a.id_producto, o.id_comercio, o.precio FROM observaciones o "
        "JOIN anuncios a USING (id_comercio, sku_comercio) WHERE a.comparable = 1 ORDER BY 1, 2, 3",
}
for nombre, sql in consultas.items():
    a, b = pd.read_sql(sql, entregada), pd.read_sql(sql, nueva)
    print(f"{nombre}: {'IGUALES' if a.equals(b) else 'DIFERENTES'} ({len(b)} filas)")

## 2. Modo en línea con un comercio (Locatel)

In [ ]:
!python scraper.py --comercios locatel --db /content/prueba_en_linea.sqlite | tail -n 4

In [ ]:
prueba = sqlite3.connect("/content/prueba_en_linea.sqlite")
pd.read_sql("SELECT id_ejecucion, id_comercio, inicio, modo, total_reportado_sitio, "
            "productos_recorridos, registros_categoria, estado FROM ejecuciones", prueba)